# 04 — Ventanado y extracción de features

Construcción de ventanas temporales alrededor de cada latido y cálculo de un conjunto inicial de features (temporales y RR).

**Notas metodológicas**

- La unidad de análisis es la ventana, anclada en el tiempo de cada latido.
- Se permite sobrelapamiento entre ventanas (configurable).
- `beat_type` **no se usa como feature**. Solo se conserva como columna informativa para inspección.
- Se mantiene `case_id` en la tabla de features para el split por grupos posterior.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config
from src.data_loading import load_metadata, load_annotations_for_case
from src.preprocessing import apply_basic_filters
from src.windowing import build_windows_for_case
from src.features import compute_time_features_batch, compute_rr_features
from src.utils import get_logger

logger = get_logger("nb04")

## 2. Selección de casos y carga de señal

In [ ]:
metadata = load_metadata()
sample_case_ids = metadata[config.CASE_ID_COLUMN].head(3).tolist()
print("case_ids:", sample_case_ids)

ecg_by_case = {}
for cid in sample_case_ids:
    cached = config.VITALDB_WAVEFORMS_DIR / f"case_{cid}.npy"
    if not cached.exists():
        raise FileNotFoundError(
            f"Falta {cached}. Ejecuta primero 03_ecg_loading_and_visualization.ipynb."
        )
    ecg_by_case[cid] = np.load(cached)

## 3. Construcción de ventanas por caso

In [ ]:
all_windows = []
all_specs = []

for cid in sample_case_ids:
    beats = load_annotations_for_case(cid)
    beats_filtered = apply_basic_filters(beats)

    if beats_filtered.empty:
        logger.warning("case_id=%s no tiene latidos tras los filtros base.", cid)
        continue

    windows, specs = build_windows_for_case(
        signal=ecg_by_case[cid],
        beats=beats_filtered,
        case_id=cid,
        fs_hz=config.DEFAULT_ECG_FS_HZ,
        window_seconds=config.DEFAULT_WINDOW_SECONDS,
        overlap=config.DEFAULT_WINDOW_OVERLAP,
    )
    logger.info("case_id=%s | ventanas=%d", cid, len(specs))
    all_windows.append(windows)
    all_specs.extend(specs)

windows_array = np.concatenate(all_windows, axis=0) if all_windows else np.empty((0, 0))
print("Total de ventanas:", windows_array.shape)

## 4. Features temporales por ventana

In [ ]:
time_features = compute_time_features_batch(windows_array)

spec_df = pd.DataFrame([
    {
        config.CASE_ID_COLUMN: s.case_id,
        "beat_index": s.beat_index,
        "start_sample": s.start_sample,
        "end_sample": s.end_sample,
        config.TARGET_COLUMN: s.label,
    }
    for s in all_specs
])

features_df = pd.concat([spec_df.reset_index(drop=True), time_features.reset_index(drop=True)], axis=1)
features_df.head()

## 5. Features RR por caso

Calculadas sobre los tiempos de latido (después de aplicar filtros de calidad y de ritmo).

In [ ]:
rr_rows = []
for cid in sample_case_ids:
    beats = apply_basic_filters(load_annotations_for_case(cid))
    if config.BEAT_TIME_COLUMN not in beats.columns:
        continue
    rr_feats = compute_rr_features(beats[config.BEAT_TIME_COLUMN].to_numpy())
    rr_feats[config.CASE_ID_COLUMN] = cid
    rr_rows.append(rr_feats)

rr_df = pd.DataFrame(rr_rows)
rr_df

## 6. Persistencia opcional en `data/processed/`

Las salidas de este notebook **no se versionan** (excluidas por `.gitignore`).

In [ ]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_features = config.PROCESSED_DIR / "features_baseline.parquet"
features_df.to_parquet(out_features, index=False)
print("Guardado:", out_features)

## 7. Próximos pasos

- Pasar a `05_baseline_modeling.ipynb` para entrenar un baseline con split por `case_id`.